<a href="https://colab.research.google.com/github/SasankaPandaSCIT/Automate-with-Gen-AI-Agents/blob/main/Module%204/4.2%20Building%20RAGs%20and%20Multi-Step%20Chains/2%20Tutorial%20-%20PromptTemplate%20and%20External%20Data%20Source%20Processors%20Openrouter.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

### OpenRouter API Key

This notebook uses OpenRouter through LangChain's OpenAI-compatible interface. Enter your OpenRouter API key when prompted. You do not need a separate OpenAI API key.


In [1]:
# Install pinned dependencies (Colab-ready; safe to re-run).
# Based on latest compatible versions
!pip install -q langchain-core==1.6.3 langchain-openai==1.6.2 langchain-pinecone==0.2.13 pinecone==7.3.0 python-dotenv==1.2.3 tiktoken==0.14.0


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 571.8/571.8 kB 13.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 125.8/125.8 kB 6.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 587.6/587.6 kB 27.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 259.3/259.3 kB 16.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 65.5/65.5 kB 4.2 MB/s eta 0:00:00


## Tutorial: PromptTemplate and Pinecone Retrieval
We’ll seed a small vector index and retrieve context to feed a clean prompt.

Learning outcomes:
- Design robust `PromptTemplate`s for grounded answers
- Set up Pinecone with OpenAI embeddings
- Retrieve top‑k docs and wire them to the Q&A chain


In [2]:
import os, time
from typing import List
from dotenv import load_dotenv
from getpass import getpass

load_dotenv()
OPENROUTER_API_KEY = os.getenv("OPENROUTER_API_KEY") or getpass("Enter OPENROUTER_API_KEY (hidden): ")
PINECONE_API_KEY = os.getenv("PINECONE_API_KEY") or getpass("Enter PINECONE_API_KEY (hidden): ")

# PineconeVectorStore reads the key from the environment rather than a variable.
os.environ["PINECONE_API_KEY"] = PINECONE_API_KEY
MODEL = "openai/gpt-4.1-mini"

# Pinecone setup
INDEX_NAME = os.getenv("PINECONE_INDEX", "lc-demo-index")

from pinecone import Pinecone, ServerlessSpec
pc = Pinecone(api_key=PINECONE_API_KEY)
if not pc.has_index(INDEX_NAME):
    pc.create_index(
        name=INDEX_NAME,
        dimension=1536,  # matches OpenAI text-embedding-3-small
        metric="cosine",
        spec=ServerlessSpec(cloud="aws", region=os.getenv("PINECONE_REGION", "us-east-1"))
    )

# A newly created index is not queryable until it reports ready.
while not pc.describe_index(INDEX_NAME).status["ready"]:
    time.sleep(1)

from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from langchain_pinecone import PineconeVectorStore
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import StrOutputParser

llm = ChatOpenAI(model=MODEL, api_key=OPENROUTER_API_KEY, base_url="https://openrouter.ai/api/v1", temperature=0, seed=42)
embeddings = OpenAIEmbeddings(model="openai/text-embedding-3-small", api_key=OPENROUTER_API_KEY, base_url="https://openrouter.ai/api/v1", dimensions=1536)
vectorstore = PineconeVectorStore(index_name=INDEX_NAME, embedding=embeddings)


Enter OPENROUTER_API_KEY (hidden): ··········
Enter PINECONE_API_KEY (hidden): ··········


### Step 1: Seed Pinecone with a few texts
We’ll upsert 2‑3 short passages to retrieve from during Q&A.


In [3]:
texts: List[str] = [
    "LangChain helps developers build LLM applications by composing prompts, chains, tools, and agents.",
    "It emphasizes modularity, integrations, and production‑ready patterns such as retrieval and evaluation.",
    "Prompt templates and retrievers make Q&A more reliable by grounding answers in context."
]
metas = [{"source": "local", "doc_id": f"demo-{i}"} for i in range(len(texts))]
vectorstore.add_texts(texts=texts, metadatas=metas)
print(f"Indexed {len(texts)} texts into Pinecone index '{INDEX_NAME}'.")


Indexed 3 texts into Pinecone index 'lc-demo-index'.


### Step 2: Design a robust PromptTemplate for retrieval
We capture source and context explicitly so outputs remain auditable.


In [4]:
ingest_template = (
    "You answer questions using the provided CONTEXT. Cite the SOURCE if helpful.\n"
    "If the answer is not present, respond: I don't know.\n\n"
    "CONTEXT:\n{context}\n\n"
    "QUESTION: {question}\n"
    "ANSWER:"
)
ingest_prompt = PromptTemplate.from_template(ingest_template)


### Step 3: Retrieve top‑k from Pinecone and run the chain
We build a context window from retrieved docs and pass it to the prompt.


In [6]:
ingest_chain = ingest_prompt | llm | StrOutputParser()
retriever = vectorstore.as_retriever(search_kwargs={"k": 4})

def build_context(query: str) -> str:
    docs = retriever.invoke(query)
    return "\n\n".join(d.page_content for d in docs)

q1 = "What is LangChain and one benefit?"
ctx1 = build_context(q1)
print(ingest_chain.invoke({
    "context": ctx1,
    "question": q1
}))


LangChain is a tool that helps developers build large language model (LLM) applications by composing prompts, chains, tools, and agents. One benefit of LangChain is that it makes Q&A more reliable by using prompt templates and retrievers to ground answers in context. (SOURCE)


### Step 4: Try another query
We’ll reuse the retriever to build a new context window.


In [8]:
q2 = "Name two components LangChain provides to developers."
ctx2 = build_context(q2)
print(ingest_chain.invoke({
    "context": ctx2,
    "question": q2
}))


Two components LangChain provides to developers are prompt templates and retrievers.
